# Gene Centric indexer
First attempt at gene indexing.
Works, although very inefficiently
```
gene{}
     |___ case[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [2]:
url = 's3a://test/small_combined_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()
        

In [3]:
#df = df.limit(500000)

In [4]:
'''
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true', comment='#', delimiter='\t')\
                .load('/home/ubuntu/tests/data/test.maf')
'''        

"\ndf = sqlContext.read.format('com.databricks.spark.csv')                .options(header='true', inferschema='true', comment='#', delimiter='\t')                .load('/home/ubuntu/tests/data/test.maf')\n"

## Rename and select desired columns in the mafs

In [5]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [6]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [7]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_uuid', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_uuid':'ssm_uuid'})
maf_ssm_map.update({'ssm_uuid':'ssm_uuid'})

## Slice and dice until we get to the format we want

### Gene df

In [8]:
gene_df = maf_df.select(*( col(k) for k in maf_gene_map.keys() + ['_case_submitter_id'] ))
# Fill in empty data we don't know about
gene_df = gene_df.withColumn('description', lit(None).cast(StringType()))\
                 .drop_duplicates()

In [9]:
#gene_df.count()

### SSM df

In [10]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()
#ssm_df.printSchema()

### Transcript-annotation df

```
transcript{}
     |_____ annotation{}
```

In [11]:
# Rename columns
tran_anno_df = maf_df.select(*( maf_transcript_map.keys() + maf_annotation_map.keys() ))
# Select annotation into nested format
tran_anno_df = tran_anno_df.select(struct(*maf_annotation_map.keys()).alias('annotation'), *maf_transcript_map.keys())\
                           .drop_duplicates()
#tran_anno_df.printSchema()

In [12]:
#tran_anno_df.count()

### Observation df

In [13]:
observation_df = maf_df.select(*(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select(struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                .drop('ssm_uuid')\
                                .drop_duplicates()

In [14]:
#observation_df.count()

### Get case dataframe from existing graph

In [15]:
'''
doc = requests.get('http://elasticsearchvis.service.consul:9200/gdc_from_graph/case/_search',
                   auth=(os.environ.get("GDC_ES_USER"),
                         os.environ.get("GDC_ES_PASS"))
                  ).json()['hits']['hits'][0]['_source']

docs = [ r['_source'] for r in requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph/case/_search?size=1',
                  ).json()['hits']['hits']]

paths = [ get_array_paths(d) for d in docs ]
paths = reduce(set.union, map(set, paths))
'''

'\ndoc = requests.get(\'http://elasticsearchvis.service.consul:9200/gdc_from_graph/case/_search\',\n                   auth=(os.environ.get("GDC_ES_USER"),\n                         os.environ.get("GDC_ES_PASS"))\n                  ).json()[\'hits\'][\'hits\'][0][\'_source\']\n\ndocs = [ r[\'_source\'] for r in requests.get(\'http://elasticsearch.service.consul:9200/gdc_from_graph/case/_search?size=1\',\n                  ).json()[\'hits\'][\'hits\']]\n\npaths = [ get_array_paths(d) for d in docs ]\npaths = reduce(set.union, map(set, paths))\n'

In [16]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*')\
    .option('es.read.field.as.array.include','')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")

In [17]:
#case_df = case_df.limit(10)

## Assemble constituent parts

Remember what we're shooting for:
```
gene{}
     |___ case[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

### Merge annotation with transcript

In [18]:
cons_tran_anno_df = tran_anno_df.select(struct(struct(*tran_anno_df.columns).alias('transcript')).alias('consequence'), 'gene_symbol')\
                                .groupBy('gene_symbol')\
                                .agg(collect_list('consequence').alias('consequence'))
                                #.drop('gene_symbol')

In [19]:
#cons_tran_anno_df.count()

### Join observation with consequence

In [20]:
#observation_df.persist().count()
#cons_tran_anno_df.persist().count()

In [21]:
import random
def salt(key, doc_count=0):
    return str(random.randint(0,int(max(0,doc_count-1024)**5)))+key
salt_udf = udf(salt, StringType())

In [22]:
salted_observation_df = observation_df.withColumn('salt_key', salt_udf(col('gene_symbol')))\
                                     #.repartition(1024, 'salt_key')
salted_observation_df.persist().count()

18710

In [23]:
salted_consequence_df = cons_tran_anno_df.withColumn('salt_key', salt_udf(col('gene_symbol')))\
                                         .repartition(64, 'salt_key')
salted_consequence_df.persist().count()

10656

In [24]:
# Partitions before and after salting
#pdf = salted_observation_df.groupBy('gene_symbol').agg(count('gene_symbol')).toPandas()
#pdf.plot()
#pdf = salted_observation_df.groupBy('salt_key').agg(count('gene_symbol')).toPandas()
#pdf.plot()

In [25]:
cons_obs = salted_consequence_df.join(salted_observation_df, salted_consequence_df.gene_symbol == salted_observation_df.gene_symbol, 'outer')\
                            .drop(salted_consequence_df.gene_symbol)\
                            .drop(salted_consequence_df.salt_key)\
                            .select('gene_symbol','consequence',struct(*[c for c in salted_observation_df.columns if c != 'gene_symbol']).alias('observation'))                   

In [26]:
#pdf = cons_obs.groupBy('gene_symbol').agg(count('gene_symbol')).toPandas()

### Join consequence into ssm

In [27]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [28]:
def view_salt(pdf1, pdf2):
    f, axes = plt.subplots(1,2, figsize=(10,4), sharex=False)
    pdf1.columns = ['salt_key', 'count']
    pdf2.columns = ['salt_key', 'count']
    axes[0].plot(pdf1['count'], 'b.')
    axes[0].set_yscale('log')
    axes[0].set_ylim(10)
    axes[0].set_xlim(0,len(pdf1))
    axes[0].set_title('unsalted')
    axes[1].plot(pdf2['count'], 'g.')
    axes[1].set_xlim(0,len(pdf2))
    axes[1].set_title('salted')
    plt.show()

In [29]:
pdf = cons_obs.select('gene_symbol', size('consequence')).toPandas()

In [30]:
pdf.max()

gene_symbol          hsa-mir-1253
size(consequence)              59
dtype: object

In [31]:
# Salt consequence-observation
salted_cons_obs = cons_obs.withColumn('doc_count', size(col('consequence')))\
                            .withColumn('salt_key', salt_udf(col('gene_symbol'), col('doc_count')))
#salted_cons_obs.repartition(1024, 'salt_key').count()

In [32]:
#pdf = salted_cons_obs.groupBy('salt_key').agg(sum('doc_count').alias('sum')).toPandas()
#pdf.sort_values('sum', ascending=False).reset_index(drop=True).plot()
#plt.gca().set_yscale('log')

In [33]:
#pdf = salted_cons_obs.groupBy('salt_key').agg(sum('doc_count').alias('sum')).toPandas()
#pdf.sort_values('sum', ascending=False).reset_index(drop=True).plot()
#plt.gca().set_yscale('log')

In [34]:
#pdf1 = cons_obs.groupBy('gene_symbol').agg(count('gene_symbol')).persist().toPandas()
#pdf2 = salted_cons_obs.groupBy('salt_key').agg(count('gene_symbol')).persist().toPandas()
#view_salt(pdf1,pdf2)

In [35]:
salted_ssm = ssm_df.withColumn('salt_key', salt_udf(col('gene_symbol')))
salted_ssm.repartition('salt_key').persist().count()

18430

In [36]:
#pdf1 = ssm_df.groupBy('gene_symbol').agg(count('gene_symbol')).persist().toPandas()
#pdf2 = salted_ssm.groupBy('salt_key').agg(count('gene_symbol')).persist().toPandas()
#view_salt(pdf1,pdf2)

In [37]:
sqlContext.sql("set spark.sql.shuffle.partitions=2048")

DataFrame[key: string, value: string]

In [38]:
salted_ssm.persist().count()

18430

In [39]:
ssm_cons = salted_cons_obs.repartition(2048,'salt_key').join(salted_ssm, salted_ssm.gene_symbol == salted_cons_obs.gene_symbol, 'left')\
                        .drop(salted_cons_obs.gene_symbol)\
                        .drop(salted_cons_obs.salt_key)\
                        .select('_case_submitter_id', struct('consequence','observation',*ssm_df.columns).alias('ssm'))\
                        .groupBy('_case_submitter_id')\
                        .agg(collect_list('ssm').alias('ssm'))

In [40]:
%%time
ssm_cons.persist().count()

CPU times: user 315 ms, sys: 128 ms, total: 443 ms
Wall time: 7.44 s


3906

### Join ssm with case

In [41]:
case_ssm = case_df.join(ssm_cons, case_df.submitter_id == ssm_cons._case_submitter_id, 'left')\
                    .drop(ssm_cons._case_submitter_id)\
                    .select('submitter_id', struct('ssm', *case_df.columns).alias('case'))

### DF Sizes
- Case:
- SSM: 31MB
- Observation: 241MB
- Consequence: 62MB
- Consequence-Observation: 6.2GB

### Benchmarking
Base dfs

case_df.persist().count()

case_df.unpersist().count()

ssm_df.persist().count()

ssm_df.unpersist().count()

observation_df.persist().count()

observation_df.unpersist().count()

cons_tran_anno_df.persist().count()

cons_tran_anno_df.unpersist().count()

#### Joined dfs

sqlContext.sql("set spark.sql.shuffle.partitions=5000")

cons_obs.persist().count()

cons_obs.unpersist().count()

ssm_cons.unpersist().count()

In [42]:
#case_ssm.persist().count()

### Join case with gene

In [43]:
gene_centric = gene_df.join(case_ssm, gene_df._case_submitter_id == case_ssm.submitter_id, 'inner')\
                    .groupBy(*gene_df.columns).agg(collect_list('case').alias('case'))

In [44]:
#gene_centric.explain()

In [45]:
#gene_centric.printSchema()

In [46]:
#gene_centric.cache()
gene_centric.persist().count()

17538

## Export df to es

#### Graph es index

In [47]:
print requests.get('http://elasticsearch.service.consul:9200/_cat/indices?v').text

health status index               pri rep docs.count docs.deleted store.size pri.store.size 
       close  gdc_from_graph_30                                                             
yellow open   gdc_legacy_graph_25  10   1   64215725            0     14.9gb         14.9gb 
yellow open   gdc_from_graph_34    10   1   38115517            0       13gb           13gb 
       close  gdc_legacy_graph_22                                                           
       close  gdc_from_graph_32                                                             
yellow open   gdc_from_graph_33    10   1   38115514            0       13gb           13gb 
yellow open   download_stats       10   1         96           21    971.3kb        971.3kb 
       close  gdc_legacy_graph_24                                                           
yellow open   gdc_legacy_graph_26  10   1   64215725            0     14.8gb         14.8gb 
yellow open   test_schema          10   1          6            0     

#### New vis index

In [48]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/indices?v').text

health status index                           uuid                   pri rep docs.count docs.deleted store.size pri.store.size
green  open   .monitoring-kibana-2-2016.11.21 Gg2lRUkbSpOJ69-so0TSpg   1   1      11061            0      7.3mb          3.7mb
green  open   .monitoring-kibana-2-2016.11.20 sYEaQ1ukStibxpHDnPI0Zg   1   1      17188            0      7.6mb          3.8mb
red    open   ssm                             otkQI7DyRAKcfwvZfb-NVQ   8   0    3627551            0     45.3gb         45.3gb
green  open   .monitoring-data-2              pt6so5oJRiCd9z0JwoJyKQ   1   1         10           13     51.1kb         25.5kb
red    open   case                            U_5pmk4dRpahgx7jXs2u5Q  10   0    3034183            0    440.6mb        440.6mb
green  open   .monitoring-es-2-2016.11.19     cpecQtt3Tx-Y1HP0BP0tpQ   1   1     253250          666    324.6mb          162mb
green  open   .kibana                         VktMnBTYQzioA_in3JbLSg   1   1          6            0    605.2kb

In [49]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/count/test?v').text

{"error":{"root_cause":[{"type":"index_not_found_exception","reason":"no such index","resource.type":"index_or_alias","resource.id":"test","index_uuid":"_na_","index":"test"}],"type":"index_not_found_exception","reason":"no such index","resource.type":"index_or_alias","resource.id":"test","index_uuid":"_na_","index":"test"},"status":404}


In [50]:
print requests.delete('http://elasticsearchvis.service.consul:9200/gdc_from_graph').json()

{u'status': 404, u'error': {u'index_uuid': u'_na_', u'index': u'gdc_from_graph', u'resource.type': u'index_or_alias', u'root_cause': [{u'index_uuid': u'_na_', u'index': u'gdc_from_graph', u'resource.type': u'index_or_alias', u'resource.id': u'gdc_from_graph', u'reason': u'no such index', u'type': u'index_not_found_exception'}], u'reason': u'no such index', u'type': u'index_not_found_exception', u'resource.id': u'gdc_from_graph'}}


In [51]:
%autoreload
from exports.mappings import GeneMapper
m = GeneMapper()
m.mapping['properties']['case'].keys()
m.mapping['dynamic'] = 'true'
m.mapping['properties']['case']['dynamic'] = 'true'
m.mapping['properties']['case']['properties']['files']['dynamic'] = 'true'
m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [ ]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/gene').json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"1s",
                "number_of_shards":20,
                "number_of_replicas":0,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "gene":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/gene', data=data).json()

{u'acknowledged': True}
{u'acknowledged': True, u'shards_acknowledged': True}


In [ ]:
%%time
# Stop index refreshing while we bulk load
gene_centric.write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', 'gene/gene')\
                    .option('es.http.timeout', '10m')\
                    .option('es.batch.size.bytes','1mb')\
                    .save('gene/gene')
            
requests.post('http://localhost:9200/gene/_refresh')
# 6min 30s to write with defaults
# 5min 37s to write with batchsize = 1mb
# 5min 13s to write with batchsize = 512kb

In [ ]:
#requests.get('http://localhost:9200/test/_search?size=5').json()['hits']['hits']

In [ ]:
test_query = {
  "query": {
    "nested": {
      "path": "case",
      "score_mode": "sum",
      "query": {
        "function_score": {
          "query": {
            "bool": {
              "must": [
                {
                "terms": {
                  "case.project.project_id": [
                    "TCGA-ACC"
                  ]
                }
                }
              ]
            }
          }
        }
      }
    }
  }
}
      
len(requests.post('http://localhost:9200/test/_search', data=json.dumps(test_query)).json()['hits']['hits'])